# 🗄️ Phase 3: SQL Analysis
## Credit Risk Analytics — Home Credit Default Risk
---
**Goal:** Load cleaned data into SQLite, run analytical SQL queries to produce default rate segmentations used in the Power BI dashboard.


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('application_clean.csv')
print(f"Dataset loaded: {df.shape}")


## 1. Load Data into SQLite

In [ ]:
# Create SQLite DB (equivalent to MySQL locally)
conn = sqlite3.connect('credit_risk.db')

# Load main table
df.to_sql('applications', conn, if_exists='replace', index=False)
print("Table 'applications' created in SQLite ✅")

# Verify
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM applications", conn)
print(result)


## 2. Query 1 — Default Rate by Income Bracket

In [ ]:
query1 = '''
SELECT 
    INCOME_BRACKET,
    COUNT(*) AS applicants,
    ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct,
    ROUND(AVG(AMT_INCOME_TOTAL), 0) AS avg_income
FROM applications
WHERE INCOME_BRACKET != ''
GROUP BY INCOME_BRACKET
ORDER BY default_rate_pct DESC
'''

default_by_income = pd.read_sql(query1, conn)
print("=== Default Rate by Income Bracket ===")
print(default_by_income.to_string(index=False))


## 3. Query 2 — Default Rate by Age Group

In [ ]:
query2 = '''
SELECT
    AGE_GROUP,
    COUNT(*) AS applicants,
    ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM applications
GROUP BY AGE_GROUP
ORDER BY default_rate_pct DESC
'''

default_by_age = pd.read_sql(query2, conn)
print("=== Default Rate by Age Group ===")
print(default_by_age.to_string(index=False))


## 4. Query 3 — Portfolio Exposure by Loan Type

In [ ]:
query3 = '''
SELECT
    NAME_CONTRACT_TYPE,
    COUNT(*) AS total_loans,
    ROUND(SUM(AMT_CREDIT) / 1000000.0, 2) AS total_credit_millions,
    ROUND(AVG(AMT_CREDIT), 0) AS avg_credit
FROM applications
GROUP BY NAME_CONTRACT_TYPE
'''

portfolio_exposure = pd.read_sql(query3, conn)
print("=== Portfolio Exposure by Loan Type ===")
print(portfolio_exposure.to_string(index=False))


## 5. Query 4 — High Risk Applicants

In [ ]:
query4 = '''
SELECT
    SK_ID_CURR,
    NAME_CONTRACT_TYPE,
    AMT_CREDIT,
    AMT_INCOME_TOTAL,
    AGE,
    AGE_GROUP,
    INCOME_BRACKET,
    ANNUITY_INCOME_RATIO,
    TARGET
FROM applications
WHERE TARGET = 1
ORDER BY AMT_CREDIT DESC
LIMIT 20
'''

high_risk = pd.read_sql(query4, conn)
print("=== Top 20 High-Risk Applicants ===")
print(high_risk.to_string(index=False))


## 6. Query 5 — Default Rate by Loan Type & Age Group (Cross-Analysis)

In [ ]:
query5 = '''
SELECT
    NAME_CONTRACT_TYPE,
    AGE_GROUP,
    COUNT(*) AS applicants,
    ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM applications
GROUP BY NAME_CONTRACT_TYPE, AGE_GROUP
ORDER BY default_rate_pct DESC
'''

cross_analysis = pd.read_sql(query5, conn)
print("=== Cross Analysis: Loan Type × Age Group ===")
print(cross_analysis.to_string(index=False))


## 7. Export Results

In [ ]:
# Export all query results for Power BI / reporting
default_by_income.to_csv('sql_default_by_income.csv', index=False)
default_by_age.to_csv('sql_default_by_age.csv', index=False)
portfolio_exposure.to_csv('sql_portfolio_exposure.csv', index=False)
high_risk.to_csv('sql_high_risk_applicants.csv', index=False)

print("All SQL results exported ✅")
print("Files: sql_default_by_income.csv, sql_default_by_age.csv,")
print("       sql_portfolio_exposure.csv, sql_high_risk_applicants.csv")

conn.close()
